[Table1] model_meta
  - model_id, country, year, brand_name, model, d_type

[Table2] voc_score
  - model_id, category, total_count, avg_sc

        ↓ JOIN

[Long DF]
        ↓ Pivot
[df_wide]
  - index: model_id
  - columns:
      {category}_score
      {category}_count

        ↓ Loop
(variable1 ∈ {all, brand_name, country, year})
(variable1_value)
(y_feature ∈ pool)
(x_features = pool - {y})

        ↓ WLS

[Output 1] model_summary_df
[Output 2] coef_summary_df

### 1. 데이터 로드 & Wide 재구조화

In [0]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import statsmodels.api as sm

spark = SparkSession.builder.getOrCreate()

# 1) 테이블 로드
df_meta = spark.table("sandbox.z_jungryo_lee.buzz_model_meta").toPandas()
df_voc  = spark.table("sandbox.z_jungryo_lee.buzz_sc_score_dataset").toPandas()

# 2) JOIN
df_long = df_voc.merge(df_meta, on="model_id", how="left")

# 3) Wide 변환
df_score = df_long.pivot_table(
    index="model_id",
    columns="category",
    values="avg_sc"
).add_suffix("_score")

df_count = df_long.pivot_table(
    index="model_id",
    columns="category",
    values="total_count"
).add_suffix("_count")

df_wide = pd.concat([df_score, df_count], axis=1).fillna(0)


In [0]:

def clean_name(s: str) -> str:
    return (
        str(s)
        .replace('.', '')
        .replace('/', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
        .replace(' ', '_')
    )

def clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df


def clean_feature_pool(feature_pool: list) -> list:
    return [clean_name(f) for f in feature_pool]

df_wide = clean_col_names(df_wide)

### 2. WLS 단일 실행 함수 (y_obs, x_obs 포함)

In [0]:
def run_wls(df_wide, y_feature, x_features):
    y_score = f"{y_feature}_score"
    y_count = f"{y_feature}_count"

    X_cols = [f"{x}_score" for x in x_features]
    X_count_cols = [f"{x}_count" for x in x_features]

    # y 기준 유효 표본
    valid = df_wide[y_count] > 0
    if valid.sum() < 5:
        return None, None

    Y = df_wide.loc[valid, y_score]
    W = df_wide.loc[valid, y_count]
    X = df_wide.loc[valid, X_cols]

    X = sm.add_constant(X)

    model = sm.WLS(Y, X, weights=W).fit()

    # --- model summary ---
    model_summary = {
        "y_feature": y_feature,
        "y_obs": int(model.nobs),
        "r_squared": model.rsquared,
        "adj_r_squared": model.rsquared_adj,
        "f_statistic": model.fvalue,
        "prob_f": model.f_pvalue,
        "log_likelihood": model.llf,
        "aic": model.aic,
        "bic": model.bic,
        # "Omnibus": model.omni,
        # "Prob_Omnibus": model.omnipv,
        # "Skew": model.skew,
        # "Kurtosis": model.kurtosis,
        # "Durbin_Watson": sm.stats.stattools.durbin_watson(model.resid),
        # "Jarque_Bera": model.jarque_bera[0],
        # "Prob_JB": model.jarque_bera[1],
        "cond_no": model.condition_number
    }

    # --- coef summary ---
    coef_rows = []
    for x in x_features:
        coef_rows.append({
            "y_feature": y_feature,
            "x_feature": x,
            "coef": model.params.get(f"{x}_score", np.nan),
            "p_value": model.pvalues.get(f"{x}_score", np.nan),
            "t_value": model.tvalues.get(f"{x}_score", np.nan),
            "x_obs": int((df_wide.loc[valid, f"{x}_count"] > 0).sum())
        })

    return model_summary, coef_rows

### 3. 상위카테고리 × y-feature 전체 루프

In [0]:
FEATURE_POOL = [
    "07_01 채널_컨텐츠 APP", "07_02 구동성_구동속도_(1)TV 전반",
    "07_02 구동성_구동속도_(2)APP_웹전반", "07_03 메뉴 구성_UI",
    "07_04 SW_OS", "07_05 컨텐츠 탐색 사용성",
    "07_06 리모컨 사용성", "07_07 리모컨 디자인",
    "07_08 음성 인식_조작", "07_09 게임 기능",
    "07_10 부가 기능", "07_11 홈 IoT",
    "07_12 모바일 연동", "07_13 광고",
    "07_20 전반적 스마트 사용성"
]

FEATURE_POOL_CLEAN = clean_feature_pool(FEATURE_POOL)

model_rows = []
coef_rows = []

for variable1 in ["all", "brand_name", "country", "year", "d_type"]:

    if variable1 == "all":
        groups = {"all": df_wide.index}
    else:
        groups = df_meta.groupby(variable1)["model_id"].apply(list).to_dict()

    for v1_value, ids in groups.items():
        df_sub = df_wide.loc[df_wide.index.intersection(ids)]
        if df_sub.shape[0] < 5:
            continue

        for y in FEATURE_POOL_CLEAN:
            x_feats = [f for f in FEATURE_POOL_CLEAN if f != y]

            model_sum, coef_sum = run_wls(df_sub, y, x_feats)
            if model_sum is None:
                continue

            model_sum["variable1"] = variable1
            model_sum["variable1_value"] = v1_value
            model_rows.append(model_sum)

            for r in coef_sum:
                r["variable1"] = variable1
                r["variable1_value"] = v1_value
                coef_rows.append(r)

### 4. 최종 산출물 DataFrame

In [0]:

model_summary_df = pd.DataFrame(model_rows)
coef_summary_df  = pd.DataFrame(coef_rows)

display(model_summary_df.head())
display(coef_summary_df.head())


In [0]:
# (예시) 기존에 만든 모델 요약 결과가 model_summary_df 라고 가정
# variable1, variable1_value -> group_dim, group_key 로 리네이밍
model_summary_df = model_summary_df.rename(columns={"variable1": "group_dim", "variable1_value": "group_key"})
# all 케이스 가독성 보정: group_key 가 NULL/빈문자면 "all"로 통일
model_summary_df.loc[model_summary_df["group_dim"] == "all", "group_key"] = "all"

# coef 테이블에도 동일 적용
coef_summary_df = coef_summary_df.rename(columns={"variable1": "group_dim", "variable1_value": "group_key"})
coef_summary_df.loc[coef_summary_df["group_dim"] == "all", "group_key"] = "all"

In [0]:

coef_summary_df["abs_coef"] = coef_summary_df["coef"].abs()

# 2) 그룹별(= 같은 분석 컨텍스트)로 abs_coef 내림차순 랭크
#    - 그룹키: group_dim, group_key, y_feature
#    - 메서드:
#        * 'dense' : 동순위는 같은 번호, 다음은 +1 (1,1,2,3,…)
#        * 'min'   : 동순위는 같은 번호, 다음은 건너뜀 (1,1,3,4,…)
#        * 'first' : 동률이면 등장 순서대로 서로 다른 번호 부여
#    - 일반적으로 드라이버 순위엔 'dense'가 가장 쓰기 좋습니다.
coef_summary_df["driver_rank"] = (
    coef_summary_df
      .groupby(["group_dim", "group_key", "y_feature"])["abs_coef"]
      .rank(method="dense", ascending=False)
      .astype(int)
)

# is_driver: p_value < 0.1 AND abs(coef) >= 0.07 → 1, else 0

PVAL_THR = 0.1
ABS_COEF_THR = 0.07

coef_summary_df["is_driver"] = np.where(
    (coef_summary_df["p_value"] < PVAL_THR) & (coef_summary_df["coef"].abs() >= ABS_COEF_THR),
    1, 0
).astype(int)

# (선택) 정렬해서 확인
coef_summary_df = coef_summary_df.sort_values(
    by=["group_dim", "group_key", "y_feature", "driver_rank", "x_feature"]
).reset_index(drop=True)

coef_summary_df.group_key.unique()


In [0]:
from datetime import datetime

# 데이터 생성일 컬럼 추가
model_summary_df["data_created_dt"] = datetime.now()
coef_summary_df["data_created_dt"] = datetime.now()

print(f"[OK] data_created_dt 추가 완료: {datetime.now()}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

spark = SparkSession.builder.getOrCreate()

def clean_name(s: str) -> str:
    return (str(s)
            .replace('.', '')
            .replace('/', '_')
            .replace('(', '')
            .replace(')', '')
            .replace('-', '_')
            .replace(' ', '_'))

def clean_col_names_pandas(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.copy()
    pdf.columns = [clean_name(c) for c in pdf.columns]
    return pdf

def save_pandas_to_delta(
    pdf: pd.DataFrame,
    table_name: str,
    mode: str = "overwrite",              # "overwrite" | "overwrite"
    partition_cols: list = None,       # 예: ["group_dim"] 또는 ["run_date"]
    merge_schema: bool = False,        # 스키마 컬럼 추가 허용
    overwrite_schema: bool = False,    # 스키마 강제 변경 (overwrite 시)
    add_row_id: bool = False,          # Pandas index 보존용 row_id 추가
    null_to_nan: bool = True           # Pandas NaN을 Spark null로 처리
):
    """
    Pandas DF를 Delta 테이블로 저장(샌드박스용). 
    - 컬럼명 클린징(기본 제공)
    - 파티션 저장/스키마 옵션 지원
    - Databricks 권장 Delta 포맷 사용
    """
    if pdf is None or len(pdf) == 0:
        print("[save_pandas_to_delta] 입력 DataFrame이 비어 있습니다. 저장을 생략합니다.")
        return

    # 1) 컬럼명 안전화
    pdf = clean_col_names_pandas(pdf)

    # 2) Pandas index를 보존하고 싶으면 row_id 컬럼으로 추가
    if add_row_id and pdf.index.name != "row_id":
        pdf = pdf.reset_index().rename(columns={"index": "row_id"})
    elif add_row_id and pdf.index.name == "row_id":
        pdf = pdf.reset_index()

    # 3) NaN → None 변환(스파크 null로 인식)
    if null_to_nan:
        pdf = pdf.where(pd.notnull(pdf), None)

    # 4) Spark DF 변환
    sdf = spark.createDataFrame(pdf)

    # 5) 저장 옵션 구성
    writer = (sdf.write
                .format("delta")
                .mode(mode))

    if merge_schema:
        writer = writer.option("mergeSchema", "true")
    if overwrite_schema:
        writer = writer.option("overwriteSchema", "true")

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    # 6) Delta 테이블로 저장
    writer.saveAsTable(table_name)
    print(f"[save_pandas_to_delta] 저장 완료 → {table_name} (mode={mode}, partition={partition_cols}, mergeSchema={merge_schema})")


# 예시 Pandas DF (model_summary_df, coef_summary_df 등)
# model_summary_df, coef_summary_df = ...

save_pandas_to_delta(
    model_summary_df,
    table_name="sandbox.z_jungryo_lee.voc_wls_model_summary",
    mode="overwrite",
    partition_cols=["group_dim"],   # 조회 패턴에 맞게 필요 시 지정
    merge_schema=True               # 실행 중 컬럼 추가 허용
)

save_pandas_to_delta(
    coef_summary_df,
    table_name="sandbox.z_jungryo_lee.voc_wls_coef_summary",
    mode="overwrite",
    partition_cols=["group_dim"],
    merge_schema=True
)

In [0]:
from pyspark.sql import functions as F

# 기존 테이블 로드
model_tbl = "sandbox.t_online_voc_analysis.voc_wls_model_summary"
coef_tbl = "sandbox.t_online_voc_analysis.voc_wls_coef_summary"

# data_created_dt 컬럼 추가 후 덮어쓰기
for tbl in [model_tbl, coef_tbl]:
    df = spark.table(tbl).withColumn("data_created_dt", F.current_timestamp())
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(tbl)
    print(f"[DONE] {tbl} - data_created_dt 컬럼 추가 완료 ({df.count():,}건)")

In [0]:
corr_rows = []

for variable1 in ["all", "brand_name", "country", "year", "d_type"]:

    if variable1 == "all":
        group_values = ["all"]
    else:
        group_values = df_meta[variable1].dropna().unique()

    for val in group_values:

        if variable1 == "all":
            pdf = df_wide.copy()
        else:
            model_ids = df_meta[df_meta[variable1] == val]["model_id"].unique()
            pdf = df_wide.loc[df_wide.index.isin(model_ids)]

        if pdf.empty:
            continue

        for y in FEATURE_POOL_CLEAN:
            y_sc = f"{y}_score"
            y_ct = f"{y}_count"

            if y_sc not in pdf.columns:
                continue

            sub = pdf[(pdf[y_ct] > 0) & pdf[y_sc].notna()]
            if sub.empty:
                continue

            w = np.sqrt(sub[y_ct])  # ✅ 권장 1

            for x in FEATURE_POOL_CLEAN:
                x_sc = f"{x}_score"
                if x_sc not in sub.columns or x == y:
                    continue

                corr = weighted_corr(sub[x_sc], sub[y_sc], w)

                corr_rows.append({
                    "group_dim": variable1,
                    "group_key": val,
                    "y_feature": y,
                    "x_feature": x,
                    "weighted_corr": corr
                })

In [0]:

corr_df = pd.DataFrame(corr_rows)

# ✅ 기존 최종 결과 테이블 (v1)
# coef_summary_df 가 이미 존재한다고 가정

final_df = coef_summary_df.merge(
    corr_df,
    on=["group_dim", "group_key", "y_feature", "x_feature"],
    how="left"
)


display(final_df)

In [0]:

spark.createDataFrame(final_df) \
     .write \
     .format("delta").option("mergeSchema", "true") \
     .mode("overwrite") \
     .saveAsTable("sandbox.t_online_voc_analysis.voc_wls_coef_summary")


In [0]:
display(final_df)